# Differentiable FEA + PINN Journey

This notebook drills into Approach A: data generation via differentiable FEA, surrogate training, validation, and generalization outcomes.



## Story Outline

1. **Imports & Config**  
   - Load numpy/pandas/torch/plotly.  
   - Path helpers pointing at `src/approach_a_pinn` & `data/results`.
2. **Geometry + Load Recap**  
   - Inline call to shared drawing helpers (reuse from master).  
   - Connect each loadcase to dataset columns.
3. **Data Generation Walkthrough**  
   - Summarize `GenerateData_Lshape_Batch.jl` + scenario rollouts.  
   - Visualize coverage from `multi_geom_training/*.csv` with histograms + scatter overlays.  
   - Provide toggles to switch between legacy vs multi-geometry corpora.
4. **Model Architecture & Training**  
   - Auto-load `TrainPINN.py` hyperparameters, render network diagram.  
   - Reconstruct training curves directly from log files (CSV).  
   - Show inference timing snippet replicating `Proof_Speedup` but referencing multi-geometry checkpoint.
5. **Validation & Generalization**  
   - Load `pinn_generalization_test.csv` + `multi_geom_model_metrics.csv`, plot MAE/MAPE bars.  
   - 3D compliance surface (Plotly) for selected loadcases predicted vs ground truth.
6. **Integration Hooks**  
   - Provide function that master & MMC notebooks can call to get surrogate predictions under specific loadcases.  
   - Document commands to retrain & refresh artifacts.



In [11]:
# L-Bracket
draw_lbracket_2d().show()
make_bracket_mesh().show()

# Ribbed Channel
ribbed_fig = make_ribbed_channel_mesh()
ribbed_fig.show()

# Tapered Plate
tapered_fig = make_tapered_plate_mesh()
tapered_fig.show()

# Load cases - use interactive widget to view (rendering all at once can be slow)
print("Use the interactive widget below to view load cases, or call:")
print("  make_geometry_with_loads_3d('l_bracket', 'horizontal_tip').show()")
print("  make_geometry_with_loads_3d('ribbed_channel', 'upward_tip').show()")
print("  etc.")

# Example: Show one load case
make_geometry_with_loads_3d("l_bracket", "horizontal_tip").show()

## Geometry & Load Recap

We start by reusing the shared geometry helpers to visualize the L-bracket and its two canonical load cases. This mirrors the master story but keeps the context close to the PINN workflow.



In [12]:
draw_lbracket_2d().show()
make_bracket_mesh().show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Multi-Geometry Dataset Walkthrough

We materialize the dataset manifest directly from the CSV corpus in `data/results/multi_geom_training`. This replaces the historical histogram PNGs with executable Plotly figures.



In [ ]:
def load_multi_dataset() -> pd.DataFrame:
    frames = []
    for path in list_multi_geom_paths():
        df = pd.read_csv(path)
        for col in SCREW_COLS:
            if col not in df.columns:
                df[col] = 0.0
        frames.append(df)
    if not frames:
        raise RuntimeError("No CSV files found in data/results/multi_geom_training")
    df = pd.concat(frames, ignore_index=True)
    return df

multi_df = load_multi_dataset()
print(f"Loaded {len(multi_df)} samples across {multi_df['geometry'].nunique()} geometries and {multi_df['load_case'].nunique()} load cases.")

agg = multi_df.groupby(["geometry", "load_case"], as_index=False).agg(
    samples=("geometry", "count"),
    compliance_min=("compliance", "min"),
    compliance_max=("compliance", "max"),
    compliance_mean=("compliance", "mean"),
)
display(agg)



Loaded 180 samples across 3 geometries and 5 load cases.


,geometry,load_case,samples,compliance_min,compliance_max,compliance_mean
0,l_bracket,horizontal_tip,30,207.432151,478.728994,373.512733
1,l_bracket,vertical_tip,30,989.692899,1546.936461,1217.775057
2,ribbed_channel,lateral_shear,40,0.619838,0.856086,0.745128
3,ribbed_channel,upward_tip,40,32.156741,32.438080,32.289303
4,tapered_plate,combined_tip,40,22.419019,23.574407,22.928877


In [ ]:
sample_bar = px.bar(
    agg,
    x="geometry",
    y="samples",
    color="load_case",
    title="Sample Count per Geometry & Load Case",
    barmode="stack",
)
sample_bar.update_layout(xaxis_title="Geometry", yaxis_title="Samples")
sample_bar.show()

hist_fig = px.histogram(
    multi_df,
    x="compliance",
    color="geometry",
    nbins=30,
    opacity=0.7,
    title="Compliance Distribution Across Geometries",
)
hist_fig.update_layout(xaxis_title="Compliance (J)")
hist_fig.show()



In [ ]:
legacy_path = RESULTS_DIR / "pinn_training_data.csv"
legacy_hist = plot_dataset_histogram(legacy_path, "Legacy Single-Geometry Dataset")
legacy_hist.show()
legacy_scatter = plot_dataset_scatter(legacy_path, "Legacy Single-Geometry Dataset")
legacy_scatter.show()



In [ ]:
scatter_fig = plot_dataset_scatter(
    RESULTS_DIR / "multi_geom_training" / "l_bracket_horizontal.csv",
    label="L-bracket Horizontal Tip",
    has_header=True,
)
scatter_fig.show()



ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Model Architecture & Training Diagnostics

We rebuild the training experiment inline: load the dataset, encode screw + geometry/load features, train a compact surrogate, and compare the results with the production checkpoint stored in `artifacts_multi_geom/`.



In [ ]:
metadata_path = PINN_DIR / "artifacts_multi_geom" / "dataset_metadata.json"
stats_path = PINN_DIR / "artifacts_multi_geom" / "norm_stats_multi_geom.npz"
model_path = PINN_DIR / "artifacts_multi_geom" / "pinn_multi_geom.pth"

metadata = json.loads(metadata_path.read_text())
prod_model = SurrogateModel(
    input_dim=metadata["input_dim"],
    hidden_size=metadata["hidden_size"],
)
prod_model.load_state_dict(torch.load(model_path, map_location="cpu"))
prod_model.eval()
param_count = sum(p.numel() for p in prod_model.parameters())

model_summary = pd.DataFrame(
    {
        "num_samples": [metadata["num_samples"]],
        "input_dim": [metadata["input_dim"]],
        "hidden_size": [metadata["hidden_size"]],
        "parameters": [param_count],
        "geometries": [", ".join(metadata["geom_names"])],
        "load_cases": [", ".join(metadata["load_cases"])],
    }
)
display(model_summary)



In [ ]:
def encode_dataset(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, Dict]:
    geom_names = sorted(df["geometry"].unique())
    load_cases = sorted(df["load_case"].unique())
    geom_map = {name: idx for idx, name in enumerate(geom_names)}
    load_map = {name: idx for idx, name in enumerate(load_cases)}

    geom_one_hot = np.eye(len(geom_names))[df["geometry"].map(geom_map)]
    load_one_hot = np.eye(len(load_cases))[df["load_case"].map(load_map)]
    screw_data = df[SCREW_COLS].values.astype(np.float32)
    X = np.hstack([screw_data, geom_one_hot, load_one_hot]).astype(np.float32)
    y = df["compliance"].values.astype(np.float32).reshape(-1, 1)
    meta = {
        "geom_names": geom_names,
        "load_cases": load_cases,
        "input_dim": X.shape[1],
    }
    return X, y, meta


def train_with_history(X: np.ndarray, y: np.ndarray, *, hidden: int = 96, epochs: int = 400, lr: float = 1e-3):
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0)
    y_mean = y.mean(axis=0)
    y_std = y.std(axis=0)
    X_std[X_std == 0] = 1.0
    y_std[y_std == 0] = 1.0

    X_norm = (X - X_mean) / X_std
    y_norm = (y - y_mean) / y_std

    X_tensor = torch.tensor(X_norm, dtype=torch.float32)
    y_tensor = torch.tensor(y_norm, dtype=torch.float32)

    model = SurrogateModel(input_dim=X_tensor.shape[1], hidden_size=hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    stats = {"X_mean": X_mean, "X_std": X_std, "y_mean": y_mean, "y_std": y_std}
    return model, stats, history


X_full, y_full, meta = encode_dataset(multi_df)
sanity_model, sanity_stats, loss_history = train_with_history(X_full, y_full, epochs=250, lr=5e-4)
print(f"Finished sanity training → final normalized MSE {loss_history[-1]:.6f}")

loss_fig = px.line(y=loss_history, title="Inline Training Loss", labels={"index": "Epoch", "value": "MSE"})
loss_fig.show()



In [ ]:
def denormalize(pred_norm: np.ndarray, stats: dict) -> np.ndarray:
    return pred_norm * stats["y_std"] + stats["y_mean"]


def evaluate_model(model: nn.Module, stats: dict, X: np.ndarray, y: np.ndarray) -> pd.Series:
    X_norm = (X - stats["X_mean"]) / stats["X_std"]
    with torch.no_grad():
        pred_norm = model(torch.tensor(X_norm, dtype=torch.float32)).numpy()
    preds = denormalize(pred_norm, stats)
    mae = np.mean(np.abs(preds - y))
    mape = np.mean(np.abs((preds - y) / y)) * 100
    r2 = 1 - np.sum((preds - y) ** 2) / np.sum((y - y.mean()) ** 2)
    return pd.Series({"MAE (J)": mae, "MAPE (%)": mape, "R^2": r2})

prod_stats_np = np.load(stats_path)
prod_stats = {k: prod_stats_np[k] for k in prod_stats_np.files}
prod_metrics = evaluate_model(prod_model, prod_stats, X_full, y_full)
sanity_metrics = evaluate_model(sanity_model, sanity_stats, X_full, y_full)

comparison = pd.DataFrame(
    {
        "Production": prod_metrics,
        "Inline Sanity": sanity_metrics,
    }
)
display(comparison)

parity_fig = px.scatter(
    x=y_full.flatten(),
    y=denormalize(
        prod_model(torch.tensor((X_full - prod_stats["X_mean"]) / prod_stats["X_std"], dtype=torch.float32)).detach().numpy(),
        prod_stats,
    ).flatten(),
    title="PINN Prediction Parity",
    labels={"x": "FEA Compliance (J)", "y": "PINN Prediction (J)"},
)
parity_fig.add_trace(
    go.Scatter(x=[y_full.min(), y_full.max()], y=[y_full.min(), y_full.max()], mode="lines", name="Ideal")
)
parity_fig.show()



## Validation & Generalization Figures

We compare the refreshed multi-geometry surrogate against the legacy single-geometry model using the curated metrics CSV, then explore the generalization samples.



In [ ]:
metrics_path = ROOT / "src/experiments/scenario_validation/results/multi_geom_model_metrics.csv"
metrics_df = pd.read_csv(metrics_path)
mae_fig = px.bar(
    metrics_df,
    x="geometry",
    y="multi_mae",
    color="load_case",
    title="Multi-Geometry PINN MAE by Scenario",
    labels={"multi_mae": "MAE (J)"},
)
mae_fig.show()

relative_fig = px.bar(
    metrics_df.melt(
        id_vars=["geometry", "load_case"],
        value_vars=["multi_mape_pct", "legacy_mape_pct"],
        var_name="model",
        value_name="MAPE (%)",
    ),
    x="geometry",
    y="MAPE (%)",
    color="model",
    facet_col="load_case",
    title="MAPE Comparison vs Legacy PINN",
)
relative_fig.update_yaxes(matches=None)
relative_fig.show()



In [ ]:
generalization_path = RESULTS_DIR / "pinn_generalization_test.csv"
gen_df = pd.read_csv(generalization_path)

# Run the production model on the same screw placements for reproducibility
geom_one_hot = np.eye(len(metadata["geom_names"]))[0]  # assume l_bracket
load_one_hot = np.eye(len(metadata["load_cases"]))[metadata["load_cases"].index("horizontal_tip")]
repeated_geom = np.repeat(geom_one_hot[None, :], len(gen_df), axis=0)
repeated_load = np.repeat(load_one_hot[None, :], len(gen_df), axis=0)
screw_data = gen_df[["s1_x", "s1_y", "s2_x", "s2_y"]].values
if screw_data.shape[1] < metadata["input_dim"] - len(geom_one_hot) - len(load_one_hot):
    padding = metadata["input_dim"] - len(geom_one_hot) - len(load_one_hot) - screw_data.shape[1]
    screw_data = np.pad(screw_data, ((0, 0), (0, padding)), mode="constant")
X_gen = np.hstack([screw_data, repeated_geom, repeated_load])
with torch.no_grad():
    preds_norm = prod_model(torch.tensor((X_gen - prod_stats["X_mean"]) / prod_stats["X_std"], dtype=torch.float32)).numpy()
    preds = denormalize(preds_norm, prod_stats)

gen_fig = px.scatter_3d(
    x=gen_df["s1_x"],
    y=gen_df["s1_y"],
    z=preds.flatten(),
    color=preds.flatten(),
    title="Generalization Samples (PINN Predictions)",
    labels={"x": "s1_x", "y": "s1_y", "z": "Compliance (J)"},
)
gen_fig.show()



NameError: name 'metadata' is not defined

## Reusable PINN Prediction Helper

Expose a small utility so other notebooks (master, MMC) can request compliance estimates without reloading weights manually.



In [ ]:
geom_index = {name: idx for idx, name in enumerate(metadata["geom_names"])}
load_index = {name: idx for idx, name in enumerate(metadata["load_cases"])}

def pinn_predict(s1_xy: Tuple[float, float], s2_xy: Tuple[float, float], *, geometry: str = "l_bracket", load_case: str = "horizontal_tip") -> float:
    geom_vec = np.eye(len(metadata["geom_names"]))[geom_index[geometry]]
    load_vec = np.eye(len(metadata["load_cases"]))[load_index[load_case]]
    screw_vec = np.array([*s1_xy, *s2_xy], dtype=np.float32)
    if len(screw_vec) < len(SCREW_COLS):
        screw_vec = np.pad(screw_vec, (0, len(SCREW_COLS) - len(screw_vec)))
    features = np.hstack([screw_vec, geom_vec, load_vec])
    with torch.no_grad():
        pred_norm = prod_model(torch.tensor(((features - prod_stats["X_mean"]) / prod_stats["X_std"]), dtype=torch.float32))
    value = float(denormalize(pred_norm.numpy(), prod_stats))
    return value

example_value = pinn_predict((20.0, 30.0), (15.0, 80.0))
print(f"Example compliance prediction: {example_value:.2f} J")



NameError: name 'metadata' is not defined

## Where to Go Next

- Jump back to `master_story.ipynb` for the comparison narrative.  
- Open `mmc_story.ipynb` once it’s ready to cross-check optimization paths against these surrogate predictions.  
- Regenerate this notebook after updating datasets or retraining the PINN so every figure remains in sync.

